# 04 - CLTV, RFM, churn y Customer 360

Objetivo: analizar la capa final de cliente creada por el DWH: valor, margen, recencia, RFM, riesgo de abandono y accion recomendada.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_NAME = os.getenv("DB_NAME", "saleshealth")
DB_USER = os.getenv("DB_USER", os.getenv("USER", "postgres"))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if DB_HOST:
    auth = f"{DB_USER}:{DB_PASSWORD}@" if DB_PASSWORD else f"{DB_USER}@"
    port = f":{DB_PORT}" if DB_PORT else ""
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2://{auth}{DB_HOST}{port}/{DB_NAME}")
else:
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2:///{DB_NAME}")

engine = create_engine(DATABASE_URL)

def q(sql):
    return pd.read_sql_query(text(sql), engine)

In [ ]:
customer_360 = q("select * from dwh.customer_360")
customer_360.shape

In [ ]:
customer_360[[
    "ingresos_netos",
    "margen_neto_estimado",
    "cltv_historico_margen",
    "cltv_estimado_12m_margen",
    "churn_risk_score",
]].describe().round(2)

In [ ]:
rfm = customer_360.groupby("segmento_rfm").agg(
    clientes=("customer_id", "count"),
    ingresos=("ingresos_netos", "sum"),
    margen=("margen_neto_estimado", "sum"),
    churn_medio=("churn_risk_score", "mean"),
).sort_values("margen", ascending=False)
rfm.round(2)

In [ ]:
ax = rfm["margen"].plot(kind="bar", figsize=(9, 4), title="Margen neto estimado por segmento RFM")
ax.set_xlabel("Segmento RFM")
ax.set_ylabel("Margen")

In [ ]:
customer_360.groupby("churn_risk_level").agg(
    clientes=("customer_id", "count"),
    margen=("margen_neto_estimado", "sum"),
    recencia_media=("recencia_dias", "mean"),
).round(2)

In [ ]:
customer_360.query("churn_risk_level in ['Critico', 'Alto']").sort_values(
    ["churn_risk_score", "margen_neto_estimado"],
    ascending=[False, False],
)[[
    "customer_id",
    "full_name",
    "recencia_dias",
    "margen_neto_estimado",
    "churn_risk_score",
    "churn_risk_level",
    "segmento_rfm",
    "recommended_action",
]].head(15)